In [1]:
from IPython.core.display import display, HTML

display(HTML("""
<style>

/* ===== Layout ===== */
.container {
    width: 100% !important;
    max-width: 2600px !important;
    margin-left: auto !important;
    margin-right: auto !important;
}

/* ===== Base Styling ===== */
body {
    background-color: #f4f5f7;
    font-family: "Inter", "Segoe UI", Roboto, sans-serif;
    font-size: 15px;
    color: #2d2d2d;
    line-height: 1.55;
}

/* ===== Code Cells ===== */
div.input_area {
    background: #ffffff !important;
    border: 1px solid #d9d9d9 !important;
    border-radius: 8px !important;
    padding: 10px !important;
}

.CodeMirror {
    font-family: "Fira Code", "Source Code Pro", monospace;
    font-size: 13.5px;
}

/* ===== OUTPUT: Clean, simple, professional ===== */
div.output_wrapper, div.output {
    background: #ffffff !important;
    border: 1px solid #e1e1e1 !important;
    border-radius: 6px !important;
    padding: 10px 14px !important;
    margin-top: 8px !important;
}

/* ===== Markdown / Text Cells ===== */
.text_cell_render {
    background: #ffffff;
    border-radius: 8px;
    padding: 18px;
    margin-bottom: 14px;
    border: 1px solid #e2e2e2;
}

/* ===== Professional Headings ===== */
h1, h2, h3, h4 {
    font-family: "Inter", sans-serif;
    font-weight: 600;
    color: #1f2a44;
}
h1 { font-size: 1.85em; border-bottom: 1px solid #d9dee7; padding-bottom: 6px; }
h2 { font-size: 1.55em; }
h3 { font-size: 1.28em; }

/* ===== Tables ===== */
table {
    border-collapse: collapse;
    width: 100%;
}
th, td {
    border: 1px solid #d7d7d7;
    padding: 8px 12px;
}
th {
    background: #eef1f5;
    font-weight: 600;
}

/* ===== Scrollbar (Minimal) ===== */
::-webkit-scrollbar { width: 7px; }
::-webkit-scrollbar-thumb {
    background: #b8c1cb;
    border-radius: 10px;
}

/* ===== Links ===== */
a {
    color: #004b9c;
}

/* ===== Cell spacing ===== */
.cell {
    margin-top: 16px;
    margin-bottom: 16px;
}

</style>
"""))


/tmp/ipykernel_4188721/2200600822.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
import  os
import fitz  # PyMuPDF
# from langgraph_utils.Info_extractor import InfoExtractorAgent
# from langgraph_utils.chat_history import SnowflakeChatMessageHistory
import json
# from langgraph_utils.file_parser import FileParser
# from langgraph_utils.digitization import main_handler
import uuid
# from langgraph_utils import creds
# from langgraph_utils.variables import PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utils.connection import get_dataiku_client_and_project
import logging
import tempfile
import re
import pymupdf4llm
from dataikuapi import DSSClient
from dataikuapi.dss.project import DSSProject
import pandas as pd
import asyncio

from crf_extraction_module.opensearch_utils import OpensearchUtil, create_embeddings_new
import nest_asyncio



# import from GLOBAL SHARED CODE
from utils import connection 
# imports from library 
# from utilities.creds import RD_PROJECT_NAME
# from variables import SECRET_NAME , TOKEN_KEY
from utils import connection 
# imports from library 
from utilities.variables import RD_PROJECT_NAME
from utilities.variables import SECRET_NAME , TOKEN_KEY
# from utilities.logging_config import logging
from IPython.core.display import display, HTML
from uuid import uuid4

/tmp/ipykernel_4188721/3899701239.py:36: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [5]:
import dataikuapi
# DATAIKU_HOST = "http://10.45.152.66:10000"
# API_SECRET_KEY = "dkuaps-b3EsRXVjU3w4y7nd4KEwibEr04CjFPZr"          
# PROJECT_NAME = "ECSGENERATION"   

# # client, proj = get_dataiku_client_and_project(PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
# client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
# proj = client.get_project(PROJECT_NAME)

DATAIKU_HOST , API_SECRET_KEY  = connection.get_dataiku_host_and_api_key(RD_PROJECT_NAME,SECRET_NAME,TOKEN_KEY)
        
client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
proj = client.get_project(RD_PROJECT_NAME)

In [6]:
# self.proj = proj
# self.client = client
s3_folder_dataset_id = proj.get_variables()['local'].get('ecs_excel') # change file upload 
input_folder = proj.get_managed_folder(s3_folder_dataset_id)
files = input_folder.list_contents()["items"]
toc_page_limit = 20
config = proj.get_variables()["local"]

In [7]:

import pandas as pd
from io import BytesIO

# Read Excel file from Dataiku managed folder
with input_folder.get_file("Historical ECS Library_new_24_11.xlsx") as stream:
    file_bytes = stream.raw.read()  # Read raw bytes

# Load both sheets
df = pd.read_excel(BytesIO(file_bytes), sheet_name="Lib - Field OIDs LLM extracted")


# Convert each sheet into list-of-dict format
data_dict = df.to_dict(orient="records")
print(data_dict[:3])



[{'id': 0, 'ecs_id': '5211a289-3d27-4319-81aa-fa29af5cbbdb', 'study': '405-201-00198', 'validation_id': 'S_EG_120_MG2', 'validation_logic': "Programmer Note: Fire when ''Not Done, specify' is entered", 'field_oids': "['EGREASND']", 'source': 'Historical', 'molecule': 'Centanafadine and Methylphenidate', 'ogcms_version': nan, 'Indication': 'Attention-deficit/hyperactivity disorder (ADHD)', 'form_oids': 'EG', 'form_name': 'EG_2 - Electrocardiogram\nEG_4 - Electrocardiogram (DM1)\nEG_1 -Electrocardiogram (D1)\nEG_6 -Electrocardiogram (D5)\nEG_7 -Electrocardiogram (D9)', 'reasoning': nan, 'action': nan, 'action_details': 'DM REVIEW: If Was ECG performed? Is no, review the If no, specify text to ensure: \n1. Text describes a valid / logical reason for not performing ECG.\n2. No spelling errors, abbreviations, special characters, etc. are present.', 'TA': 'Neurology'}, {'id': 1, 'ecs_id': '4ccea4df-4b6f-48b2-b2f1-9a46207bd321', 'study': '405-201-00198', 'validation_id': 'S_EG_142_MG1', 'vali

In [8]:
df.columns

Index(['id', 'ecs_id', 'study', 'validation_id', 'validation_logic',
       'field_oids', 'source', 'molecule', 'ogcms_version', 'Indication',
       'form_oids', 'form_name', 'reasoning', 'action', 'action_details',
       'TA'],
      dtype='object')

In [9]:
generic_mapping = []

for record in data_dict:

    generic_mapping.append({
        "ecs_id": str(record.get('ecs_id', '') or '').strip(),
        "validation_id": str(record.get('validation_id', '') or '').strip(),
        "indication" : str(record.get('Indication','') or '').strip(),
        "ogcms_version" : str(record.get('ogcms_version','') or '').strip(),
        "molecule" : str(record.get('molecule','') or '').strip(),
        "form_name": str(record.get('form_name','') or '').strip(),
        "form_domain_name":  str(record.get('form_oids','') or '').strip(),
        "field_oids" : str(record.get('field_oids','') or '').strip(),
        "validation_logic": str(record.get('validation_logic', '') or '').strip(),
        "reasoning": str(record.get("reasoning", '') or '').strip(),
        "action": str(record.get('action', '') or '').strip(),
        "action_details": str(record.get('action_details', '') or '').strip(),
        "source": str(record.get('source', '') or '').strip(),
        "ta": str(record.get('TA', '') or '').strip(),
        "path": str(record.get('study', '') or '').strip()
    })

print(f"✅ Total records created: {len(generic_mapping)}")

✅ Total records created: 4828


In [10]:
import pandas as pd
import nest_asyncio
import asyncio

nest_asyncio.apply()

form_name_list = [i["form_name"] for i in generic_mapping]
print(len(form_name_list))

# ---- CONFIG ----
BATCH_SIZE = 200   # adjust to your load
# ----------------


async def embed_in_batches(proj, items, model_id, batch_size=BATCH_SIZE):
    all_embeddings = []

    for i in range(0, len(items), batch_size):
        batch = items[i:i + batch_size]
        print(f"Embedding batch {i // batch_size + 1} / {len(items) // batch_size + 1} "
              f"(size={len(batch)})")

        try:
            emb = await create_embeddings_new(proj, batch, model_id)
            
            all_embeddings.extend(emb["response"])

        except Exception as e:
            print(f"❌ Error embedding batch starting at index {i}: {e}")
            # you can continue or break
            # continue

    return all_embeddings


if form_name_list:
    model_id = proj.get_variables()['local'].get('default_embeddings_model_id')
    form_embeddings = asyncio.run(embed_in_batches(proj, form_name_list, model_id))


4828
Embedding batch 1 / 25 (size=200)


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/pydantic/_internal/_config.py:341: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)


Embedding batch 2 / 25 (size=200)
Embedding batch 3 / 25 (size=200)
Embedding batch 4 / 25 (size=200)
Embedding batch 5 / 25 (size=200)
Embedding batch 6 / 25 (size=200)
Embedding batch 7 / 25 (size=200)
Embedding batch 8 / 25 (size=200)
Embedding batch 9 / 25 (size=200)
Embedding batch 10 / 25 (size=200)
Embedding batch 11 / 25 (size=200)
Embedding batch 12 / 25 (size=200)
Embedding batch 13 / 25 (size=200)
Embedding batch 14 / 25 (size=200)
Embedding batch 15 / 25 (size=200)
Embedding batch 16 / 25 (size=200)
Embedding batch 17 / 25 (size=200)
Embedding batch 18 / 25 (size=200)
Embedding batch 19 / 25 (size=200)
Embedding batch 20 / 25 (size=200)
Embedding batch 21 / 25 (size=200)
Embedding batch 22 / 25 (size=200)
Embedding batch 23 / 25 (size=200)
Embedding batch 24 / 25 (size=200)
Embedding batch 25 / 25 (size=28)


In [13]:
len(form_embeddings),len(generic_mapping)
for k,embeddigs in zip(generic_mapping,form_embeddings):
    k["form_name_vector"] = embeddigs
print(len(form_embeddings),len(generic_mapping))

4828 4828


In [14]:
import json
from opensearchpy.helpers import bulk
from opensearchpy import OpenSearch
opensearch_client = OpensearchUtil(client, proj)

def bulk_insert( index_name, documents):
    project = proj
        
    # Retrieve credentials from Opensearch connection 
    project_configs = project.get_variables()["local"]
    opensearch_connection = project_configs["opensearch_connection"]



    conn_info = client.get_connection(opensearch_connection).get_info()

    opensearchclient = OpenSearch(
            hosts=[{"host": conn_info["params"]["host"], "port": conn_info["params"]["port"]}],
            http_auth = (conn_info["params"]["username"], conn_info["params"]["password"]),
            use_ssl=conn_info["params"]["ssl"],
            verify_certs=False
    )
    

    docs_to_store = []
    for document in documents:

        action = {
            "_op_type": "index",  # Operation type (index = insert)
            "_index": index_name,  # Index name
            "_id": document['ecs_id'],  # Use the unique document ID
            "_source": document  # Document body
        }
        docs_to_store.append(action)

    success, failed = bulk(opensearchclient, docs_to_store)
    print(f"Successfullly indexed {success} documents.")
    print(f"Failed to index {failed} documents.")
    if failed:
        raise Exception("Document indexing encountered an unknown error. ")


index_name = proj.get_variables()['local'].get('ecs_opensearch')
# index_name = "${projectKey}_ecs_index"
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    print("index_name",index_name)
bulk_insert(index_name, generic_mapping)

index_name ecsgeneration_ecs_index


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-dataiku-genai-prod-design-xvmf2xyqsfsqod7jv3o6ggqqpm.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-dataiku-genai-prod-design-xvmf2xyqsfsqod7jv3o6ggqqpm.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTT

Successfullly indexed 4828 documents.
Failed to index [] documents.
